In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from pytorch_tabnet.tab_model import TabNetRegressor
import warnings
warnings.filterwarnings("ignore")
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import matplotlib.pyplot as plt

# 1. 加载数据
print("正在加载数据...")
# 修正文件路径，使用相同的平滑数据
df_original = pd.read_excel('../../data/pinghua_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

# 6. 检查是否有 GPU 可用
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

# 修改搜索空间定义部分
dimensions = [
    Integer(16, 128, name='n_d'),  # 决策层维度
    Integer(16, 128, name='n_a'),  # 注意力层维度
    Integer(3, 10, name='n_steps'),  # 步数
    Real(1.0, 2.0, name='gamma'),  # 稀疏度
    Real(0.0001, 0.1, name='lambda_sparse'),  # 额外的稀疏度
    Real(0.0005, 0.1, name='lr'),  # 学习率
    Integer(32, 256, name='batch_size'),  # 修改batch_size范围为2的幂次方
    Integer(16, 128, name='virtual_batch_size')  # 修改virtual_batch_size范围
]

# 修改目标函数
@use_named_args(dimensions)
def objective(n_d, n_a, n_steps, gamma, lambda_sparse, lr, batch_size, virtual_batch_size):
    # 确保batch_size和virtual_batch_size是2的幂次方
    batch_size = 2 ** int(np.log2(batch_size))  # 转换为最接近的2的幂
    virtual_batch_size = min(batch_size, 2 ** int(np.log2(virtual_batch_size)))  # 确保不大于batch_size
    
    # 创建TabNet模型
    model = TabNetRegressor(
        n_d=n_d,
        n_a=n_a,
        n_steps=n_steps,
        gamma=gamma,
        lambda_sparse=lambda_sparse,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=lr),
        scheduler_params={"step_size":50, "gamma":0.8},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax',
        device_name=device,
        verbose=0,
        seed=42
    )
    
    try:
        # 训练模型
        model.fit(
            X_train=X_train_scaled,
            y_train=y_train_scaled,
            eval_set=[(X_test_scaled, y_test_scaled)],
            eval_name=['val'],
            eval_metric=['rmse'],
            max_epochs=100,
            patience=20,
            batch_size=int(batch_size),  # 确保是整数
            virtual_batch_size=int(virtual_batch_size),  # 确保是整数
            num_workers=0
        )
        
        # 在测试集上预测
        y_pred_scaled = model.predict(X_test_scaled)
        y_pred = scaler_y.inverse_transform(y_pred_scaled)
        
        # 计算R²得分
        r2_scores = []
        for idx in range(len(target_vars)):
            r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
            r2_scores.append(r2)
        
        mean_r2 = np.mean(r2_scores)
        
        # 记录当前迭代的参数和结果
        print(f"参数: n_d={n_d}, n_a={n_a}, steps={n_steps}, gamma={gamma:.2f}, "
              f"lambda={lambda_sparse:.5f}, lr={lr:.5f}, "
              f"batch={batch_size}, vbatch={virtual_batch_size} - 平均R²: {mean_r2:.4f}")
        
        return -mean_r2
        
    except Exception as e:
        print(f"出现错误: {str(e)}")
        return 0
    
# 9. 执行贝叶斯优化
print("\n开始基于测试集R²的TabNet贝叶斯参数优化...")
result = gp_minimize(
    objective,
    dimensions=dimensions,
    n_calls=30,  # 迭代次数
    n_random_starts=10,  # 随机搜索起点数
    random_state=42,
    verbose=True
)

# 10. 提取最佳参数
best_params = {
    'n_d': result.x[0],
    'n_a': result.x[1],
    'n_steps': result.x[2],
    'gamma': result.x[3],
    'lambda_sparse': result.x[4],
    'lr': result.x[5],
    'batch_size': result.x[6],
    'virtual_batch_size': result.x[7]
}

# 在最优参数打印之后，训练最终模型之前
print("\n最优参数:")
for param, value in best_params.items():
    print(f"{param}: {value}")

# 转换batch_size和virtual_batch_size为2的幂次方
best_params['batch_size'] = 2 ** int(np.log2(best_params['batch_size']))
best_params['virtual_batch_size'] = min(best_params['batch_size'], 
                                        2 ** int(np.log2(best_params['virtual_batch_size'])))

print("\n转换后的最优参数:")
print(f"batch_size: {best_params['batch_size']}")
print(f"virtual_batch_size: {best_params['virtual_batch_size']}")

# 11. 使用转换后的最优参数创建新模型
print("\n用最优参数训练最终模型...")
best_model = TabNetRegressor(
    n_d=best_params['n_d'],
    n_a=best_params['n_a'],
    n_steps=best_params['n_steps'],
    gamma=best_params['gamma'],
    lambda_sparse=best_params['lambda_sparse'],
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=best_params['lr']),
    scheduler_params={"step_size":50, "gamma":0.8},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',
    device_name=device,
    verbose=1,
    seed=42
)

# 12. 用原始训练数据训练最终模型
best_model.fit(
    X_train=X_train_scaled,
    y_train=y_train_scaled,
    eval_set=[(X_test_scaled, y_test_scaled)],
    eval_name=['val'],
    eval_metric=['rmse'],
    max_epochs=200,  # 增加优化模型的训练轮数
    patience=30,
    batch_size=best_params['batch_size'],
    virtual_batch_size=best_params['virtual_batch_size'],
    num_workers=0
)

print("模型训练完成！")

# 13. 在测试集上进行预测
y_pred_scaled = best_model.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# 14. 评估最终模型性能
print("\n优化后TabNet模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0
min_r2 = 1.0
min_r2_var = ""

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 记录最小R²值及其变量
    if r2 < min_r2:
        min_r2 = r2
        min_r2_var = name
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 15. 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")
print(f"最小 R²: {min_r2:.4f} (变量: {min_r2_var})")

# 16. 输出贝叶斯优化的最佳结果
print("\n贝叶斯优化最佳结果:")
print(f"最佳平均R²: {-result.fun:.4f}")  # 去掉负号，因为我们优化时是负R²
print(f"优化迭代次数: {len(result.func_vals)}")
